In [10]:
import requests
from config import (
    POP_FLOW_BASE_URL,
    POP_FLOW_SERVICE_KEY,
    POP_FLOW_COLUMNS,
    START_DATE,
    END_DATE,
    BATCH_MONTHS,
    DB_URL)
import pandas as pd
import xml.etree.ElementTree as ET
from sqlalchemy import create_engine, text

In [2]:
def create_db_engine():
    return create_engine(DB_URL)

def read_districts_csv(path):
    df = pd.read_csv(path)

    required = {"Districts","Code"}

    if not required.issubset(df.columns):
        raise ValueError("CSV missing required columns.")

    df["Districts"] = df["Districts"].str.strip()
    df["Code"] = pd.to_numeric(df["Code"], errors="raise")

    return dict(zip(df['Districts'],df['Code']))

def return_df(root):
    items = root.findall(".//item")
    data = []

    for item in items:
        row = {}

        for child in item:
            if child.tag in POP_FLOW_COLUMNS:
                row[child.tag] = child.text

        data.append(row)

    return pd.DataFrame(data)

def rename_columns(df):
    return df.rename(columns={
    "statsYm":"date",
    "mvinCtpvNm":"from_province",
    "mvtCtpvNm":"to_province",
    "mvinSggNm":"from_district",
    "mvtSggNm":"to_district",
    "totNmprCnt":"total_people",
    "maleNmprCnt":"male",
    "femlNmprCnt":"female"})


In [3]:
def get_population_flow(districts,start_date,end_date):
    dfs = []
    completed = 0
    total = len(districts) ** 2
    for origin in districts:
        for destination in districts:
            param = {
                "serviceKey": POP_FLOW_SERVICE_KEY,
                "mvinAdmmCd": districts[origin],
                "mvtAdmmCd": districts[destination],
                "srchFrYm": start_date,
                "srchToYm": end_date,
                "lv": 2,
                "type":"XML",
                "numOfRows":BATCH_MONTHS,
                "pageNo":1
            }

            response = requests.get(
                POP_FLOW_BASE_URL,
                params=param,
                timeout=30)
            response.raise_for_status()
            
            completed += 1
            print(f"{completed}/{total}: Successfully loaded {origin} to {destination}")

            root = ET.fromstring(response.text)

            df = rename_columns(return_df(root))

            dfs.append(df)

    return pd.concat(dfs,ignore_index=True)



In [ ]:

# districts = read_districts_csv('/workspaces/korea-real-estate-population-movement/data/seoul_district_codes.csv')
# start_date = 202301
# end_date = 202303

# seoul_population_flow = get_population_flow(districts,start_date,end_date)
# seoul_population_flow


In [ ]:
# seoul_population_flow.info()
# seoul_population_flow['date'].unique()
# seoul_population_flow.groupby(['from_district','to_district']).size().value_counts()

In [7]:
seoul_population_flow= pd.read_csv("/workspaces/korea-real-estate-population-movement/data/seoul_population_flow_202301_202303.csv")
seoul_population_flow = seoul_population_flow.drop(columns=["Unnamed: 0"])

In [8]:
seoul_population_flow

,date,from_province,to_province,from_district,to_district,total_people,male,female
0,202301,서울특별시,서울특별시,중구,중구,176,96,80
1,202302,서울특별시,서울특별시,중구,중구,163,93,70
2,202303,서울특별시,서울특별시,중구,중구,243,129,114
3,202301,서울특별시,서울특별시,중구,종로구,30,14,16
4,202302,서울특별시,서울특별시,중구,종로구,45,19,26
...,...,...,...,...,...,...,...,...
1870,202302,서울특별시,서울특별시,강동구,송파구,488,218,270
1871,202303,서울특별시,서울특별시,강동구,송파구,444,219,225
1872,202301,서울특별시,서울특별시,강동구,강동구,1187,582,605
1873,202302,서울특별시,서울특별시,강동구,강동구,1654,819,835


In [ ]:
engine = create_engine(DB_URL)

In [21]:
create_seoul_population_flow_table_sql = '''
CREATE TABLE IF NOT EXISTS seoul_population_flow (
    date INT,
    from_province VARCHAR(50),
    to_province VARCHAR(50),
    from_district VARCHAR(50),
    to_district VARCHAR(50),
    total_people INT,
    male INT,
    female INT,
    PRIMARY KEY (date, from_province, to_province, from_district, to_district)
)
'''

insert_seoul_population_flow_sql = '''
INSERT INTO seoul_population_flow (
    date,
    from_province,
    to_province,
    from_district,
    to_district,
    total_people,
    male,
    female
) VALUES (
    :date,
    :from_province,
    :to_province,
    :from_district,
    :to_district,
    :total_people,
    :male,
    :female
)
ON CONFLICT (date, from_province, to_province, from_district, to_district) DO NOTHING
'''



In [ ]:
#Create seoul population flow table in DB
# with engine.begin() as conn:
#     conn.execute(text(create_seoul_population_flow_table_sql))


In [22]:
def main():

    df = pd.read_csv("/workspaces/korea-real-estate-population-movement/data/seoul_population_flow_202301_202303.csv")
    df = df.drop(columns=["Unnamed: 0"])

    records = df.to_dict(orient='records')

    batch_size = 100

    with engine.begin() as conn:
       for i in range(0,len(records),batch_size):
          batch = records[i:i + batch_size]
          conn.execute(text(insert_seoul_population_flow_sql),batch)

          print(f"{min(i + batch_size,len(records))}/{len(records)} inserted")

main()


100/1875 inserted
200/1875 inserted
300/1875 inserted
400/1875 inserted
500/1875 inserted
600/1875 inserted
700/1875 inserted
800/1875 inserted
900/1875 inserted
1000/1875 inserted
1100/1875 inserted
1200/1875 inserted
1300/1875 inserted
1400/1875 inserted
1500/1875 inserted
1600/1875 inserted
1700/1875 inserted
1800/1875 inserted
1875/1875 inserted
